# Inference and evaluation

In [ ]:
import os
import sys
from pathlib import Path
import re
import json
from typing import List, Tuple, Callable, OrderedDict, Dict, Any

sys.path.append(os.path.abspath(".."))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rich.progress import (
    Progress,
    TextColumn,
    BarColumn,
    TimeElapsedColumn,
    TimeRemainingColumn,
    TaskID,
)
from rich.table import Table
from rich.console import Console
from rich.box import HORIZONTALS

from sklearn import metrics

import torch
import torch.nn as nn
import torchmetrics as tm
from timm.models.resnet import resnet18, resnet34, resnet50, resnet101, resnet152
from timm.models.convnext import (
    convnext_base,
    convnext_small,
    convnext_tiny,
)
from timm.models.efficientnet import (
    efficientnet_b0,
    efficientnet_b1,
    efficientnet_b2,
    efficientnet_b3,
    efficientnet_b4,
    efficientnet_b5,
    efficientnet_b6,
    efficientnet_b7,
    efficientnetv2_s,
    efficientnetv2_m,
)
from timm.models.swin_transformer import (
    swin_tiny_patch4_window7_224,
    swin_small_patch4_window7_224,
    swin_base_patch4_window7_224,
)
from timm.models.vision_transformer import vit_base_patch16_224
from timm.models.mobilenetv3 import (
    mobilenetv4_conv_medium,
    mobilenetv4_conv_small,
    mobilenetv4_hybrid_medium,
    mobilenetv4_conv_large,
    mobilenetv4_hybrid_large,
)
from timm.models.mobilevit import (
    mobilevitv2_050,
    mobilevitv2_075,
    mobilevitv2_100,
    mobilevitv2_125,
    mobilevitv2_150,
    mobilevitv2_175,
    mobilevitv2_200,
)


from src.data import SeedRGBModule

plt.style.use("default")
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.family"] = ["serif"]
plt.rcParams["font.serif"] = ["Times New Roman"]

SEEDS = [0, 21, 42, 84, 168, 336, 672, 1344, 2688, 3407]

OVERWRITE = False
PRETRAINED = True  # False, True

BASE_DIR = Path(os.path.abspath("."))
PROJ_DIR = BASE_DIR.parent

DATA_DIR = BASE_DIR / "data"

LOGS_DIR = PROJ_DIR / "logs" / "repeat"
REPORT_DIR = DATA_DIR / f"pretrained_{PRETRAINED}" / "report"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_DIR = PROJ_DIR / "data" / "dataset" / "seeds_rgb"
path_train = DATASET_DIR / "cls_train_656_rgb.json"
path_val = DATASET_DIR / "cls_val_656_rgb.json"
path_test = DATASET_DIR / "cls_test_656_rgb.json"

BATCH_SIZE = 512

IN_CHANNELS = 3
NUM_CLASSES = 656
DEVICE = "cuda:1"
# DEVICE = "cpu"


MODELS = {
    "resnet": ["18", "34", "50"],
    "vit": ["b"],
    "swin": ["tiny"],
    "convnext": ["tiny"],
    "mobilenet": [
        "conv_small",
        "conv_medium",
        "hybrid_medium",
        "conv_large",
        "hybrid_large",
    ],
    "mobilevit": [
        "050",
        "100",
        "150",
    ],
}


## Load dataset

In [2]:
ds_module = SeedRGBModule(
    path_train=str(path_train),
    path_val=str(path_val),
    path_test=str(path_test),
    base_dirs=[str(DATASET_DIR / "images"), str(DATASET_DIR / "images_with_bg")],
    base_dirs_sampling=("uniform", "all", "all"),
    batch_size=BATCH_SIZE,
    norm_mean=[0.34865115, 0.29936219, 0.25752143],
    norm_std=[0.2302609, 0.19996711, 0.17874425],
    max_size=(224, 224),
    num_workers=16,
    persistent_workers=True,
    prefetch_factor=2,
)
ds_module.setup("test")

train_loader = ds_module.train_dataloader()
val_loader = ds_module.val_dataloader()
test_loader = ds_module.test_dataloader()

## Inference for 10 times

In [3]:
ARCHS = {
    "resnet_18": resnet18,
    "resnet_34": resnet34,
    "resnet_50": resnet50,
    "resnet_101": resnet101,
    "resnet_152": resnet152,
    "efficientnet_b0": efficientnet_b0,
    "efficientnet_b1": efficientnet_b1,
    "efficientnet_b2": efficientnet_b2,
    "efficientnet_b3": efficientnet_b3,
    "efficientnet_b4": efficientnet_b4,
    "efficientnet_b5": efficientnet_b5,
    "efficientnet_b6": efficientnet_b6,
    "efficientnet_b7": efficientnet_b7,
    "efficientnetv2_s": efficientnetv2_s,
    "efficientnetv2_m": efficientnetv2_m,
    "swin_tiny": swin_tiny_patch4_window7_224,
    "swin_small": swin_small_patch4_window7_224,
    "swin_base": swin_base_patch4_window7_224,
    "convnext_tiny": convnext_tiny,
    "convnext_small": convnext_small,
    "convnext_base": convnext_base,
    "vit_b": vit_base_patch16_224,
    "mobilevit_050": mobilevitv2_050,
    "mobilevit_075": mobilevitv2_075,
    "mobilevit_100": mobilevitv2_100,
    "mobilevit_125": mobilevitv2_125,
    "mobilevit_150": mobilevitv2_150,
    "mobilevit_175": mobilevitv2_175,
    "mobilevit_200": mobilevitv2_200,
    "mobilenet_conv_small": mobilenetv4_conv_small,
    "mobilenet_conv_medium": mobilenetv4_conv_medium,
    "mobilenet_hybrid_medium": mobilenetv4_hybrid_medium,
    "mobilenet_conv_large": mobilenetv4_conv_large,
    "mobilenet_hybrid_large": mobilenetv4_hybrid_large,
}


def load_ckpt(ckpt_path: str) -> OrderedDict:
    state_dict = torch.load(ckpt_path, map_location="cpu", weights_only=False)["state_dict"]
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        newk: str = k.replace("model.", "")  # lightning module
        newk = newk.replace("_orig_mod.", "")  # torch.compile OptimizedModule
        new_state_dict[newk] = v
    return new_state_dict


def load_model(
    name: str,
    ckpt_path: str = "",
    in_channels: int = 3,
    num_classes: int = 656,
    compile: bool = False,
) -> torch.nn.Module | Any:
    func: Callable[..., nn.Module] = ARCHS.get(name, None)  # type: ignore
    assert func is not None, f"Unknown model name: {name}"
    model = func(in_chans=in_channels, num_classes=num_classes)
    if os.path.exists(ckpt_path):
        state_dict = load_ckpt(ckpt_path)
        model.load_state_dict(state_dict)
    if compile:
        model = torch.compile(model)
    return model


METRICS: Dict[str, tm.Metric] = {
    # "accuracy": tm.Accuracy(task="multiclass", num_classes=NUM_CLASSES),
    # "precision": tm.Precision(task="multiclass", num_classes=NUM_CLASSES),
    # "recall": tm.Recall(task="multiclass", num_classes=NUM_CLASSES),
    # "f1-score": tm.F1Score(task="multiclass", num_classes=NUM_CLASSES),
    # "roc": tm.ROC(task="multiclass", num_classes=NUM_CLASSES),
    # "auroc": tm.AUROC(task="multiclass", num_classes=NUM_CLASSES),
    # "pr_curve": tm.PrecisionRecallCurve(task="multiclass", num_classes=NUM_CLASSES),
}


@torch.no_grad()
def infer_model(
    model: torch.nn.Module,
    device: str = DEVICE,
    progress: Progress | None = None,
    batch_tqdm: TaskID | None = None,
) -> Tuple[torch.Tensor, torch.Tensor]:
    model.to(device)
    model.eval()
    y_true = torch.Tensor().to(torch.long).to(device)
    y_pred = torch.Tensor().to(torch.long).to(device)

    n_total = len(val_loader) + len(test_loader)
    # for loader in [val_loader, test_loader]:
    n_total = len(test_loader)
    for loader in [test_loader]:
        for batch in loader:
            if progress and batch_tqdm:
                progress.update(batch_tqdm, advance=1, total=n_total)

            images: torch.Tensor
            labels: torch.Tensor
            images, labels = batch
            images = images.to(device)
            labels = labels.to(device)

            with torch.no_grad(), torch.autocast(device_type=device, dtype=torch.bfloat16):
                logits: torch.Tensor = model(images)
            for metric in METRICS.values():
                metric.update(preds=logits.float().cpu(), target=labels.cpu())
            preds = logits.argmax(dim=-1)

            y_true = torch.cat([y_true, labels])
            y_pred = torch.cat([y_pred, preds])

    if progress and batch_tqdm:
        progress.reset(batch_tqdm)

    return (y_true.cpu().detach(), y_pred.cpu().detach())


def find_ckpt(name: str, version: str, seed: int, pretrained: bool) -> str:
    ckpt_pattern = f"pretrained_{pretrained}/{name}/{version}/{seed}/checkpoints"
    ckpts = list((LOGS_DIR / ckpt_pattern).glob(f"{name}_{version}*.ckpt"))
    assert len(ckpts) > 0, f"not found checkpoints in {ckpt_pattern}"
    if len(ckpts) == 1:
        return str(ckpts[0])
    scores0 = [
        float(r[0].split("=")[1]) if (r := re.findall(r"val_acc=\d\.\d+", s.name)) else 0
        for s in ckpts
    ]
    ckpt_path = str(ckpts[np.argmax(scores0)])
    return ckpt_path


def test_seed_ckpt_exists(name: str, version: str, seed: int, pretrained: bool) -> bool:
    ckpt_pattern = f"pretrained_{pretrained}/{name}/{version}/{seed}/checkpoints"
    ckpts = list((LOGS_DIR / ckpt_pattern).glob(f"{name}_{version}*.ckpt"))
    return len(ckpts) > 0


def run_one_model_version(
    model_name: str,
    model_version: str,
    progress: Progress | None = None,
    seed_tqdm: TaskID | None = None,
    batch_tqdm: TaskID | None = None,
    overwrite: bool = False,
    pretrained: bool = False,
):
    if progress and seed_tqdm:
        progress.reset(seed_tqdm)

    for seed in SEEDS:
        if progress and seed_tqdm:
            progress.advance(seed_tqdm)

        if not test_seed_ckpt_exists(model_name, model_version, seed, pretrained):
            print(f"{model_name}/{model_version}/{seed} have not trained.")
            continue

        report_dir = REPORT_DIR / model_name / model_version
        if not report_dir.exists():
            report_dir.mkdir(parents=True)
        report_path = report_dir / f"{seed}.json"

        if not overwrite and report_path.exists():
            # print(f"{report_path} already exists")
            continue

        ckpt_path = find_ckpt(model_name, model_version, seed, pretrained)

        model = load_model(
            f"{model_name}_{model_version}",
            ckpt_path=ckpt_path,
            in_channels=IN_CHANNELS,
            num_classes=NUM_CLASSES,
        )

        y_true, y_pred = infer_model(
            model,
            device=DEVICE,
            progress=progress,
            batch_tqdm=batch_tqdm,
        )

        # report = classification_report(
        #     y_true=y_true,
        #     y_pred=y_pred,
        #     # target_names=ds_module.ds_train.class_names,
        #     output_dict=True,
        # )
        report = {
            "model_name": model_name,
            "model_version": model_version,
            "seed": seed,
            "y_true": y_true.tolist(),
            "y_pred": y_pred.tolist(),
            "class_names": ds_module.ds_train.class_names,
        }

        with open(report_path, "w", encoding="utf-8") as f:
            json.dump(report, f, ensure_ascii=False, indent=4)

        # cfm = metrics.confusion_matrix(y_true, y_pred)

### Save classification report

In [4]:
with Progress(
    TextColumn("[progress.description]{task.description}"),
    BarColumn(),
    TextColumn(
        "[progress.percentage]{task.completed:.0f}/{task.total:.0f} {task.percentage:>3.0f}%"
    ),
    TimeRemainingColumn(),
    TimeElapsedColumn(),
) as progress:
    model_tqdm = progress.add_task(description="models", total=len(MODELS))
    version_tqdm = progress.add_task(description="versions", total=0)
    seed_tqdm = progress.add_task(description="seeds", total=len(SEEDS))
    batch_tqdm = progress.add_task(description="batches", total=0)

    for model_name in MODELS:
        progress.advance(model_tqdm)
        progress.reset(version_tqdm)
        for model_version in MODELS[model_name]:
            progress.update(version_tqdm, total=len(MODELS[model_name]), advance=1)

            run_one_model_version(
                model_name,
                model_version,
                progress,
                seed_tqdm,
                batch_tqdm,
                overwrite=OVERWRITE,
                pretrained=PRETRAINED,
            )


Output()

## Read all report and make tables

In [3]:
"""
[
    [model_name, model_version, seed, y_true, y_pred],
    ["resnet", "18", 0, 1, 1],
]
"""
report_collection: List[List[str | int]] = []

report_models = [p for p in REPORT_DIR.glob("*") if p.is_dir()]
for m in report_models:
    report_versions = [v for v in m.glob("*") if v.is_dir()]
    for version in report_versions:
        reports = version.glob("*.json")
        for report in reports:
            with open(report, "r", encoding="utf-8") as f:
                """
                {
                    "model_name": model_name,
                    "model_version": model_version,
                    "seed": seed,
                    "y_true": y_true.tolist(),
                    "y_pred": y_pred.tolist(),
                }
                """
                data = json.load(f)
            seed = report.name.replace(".json", "")
            report_collection.extend(
                [
                    [
                        data["model_name"],
                        data["model_version"],
                        data["seed"],
                        yt,
                        yp,
                    ]
                    for yt, yp in zip(data["y_true"], data["y_pred"])
                ]
            )

In [4]:
df = pd.DataFrame(
    report_collection,
    columns=["model_name", "model_version", "seed", "y_true", "y_pred"],
)

df.head()

,model_name,model_version,seed,y_true,y_pred
0,swin,tiny,3407,244,244
1,swin,tiny,3407,237,237
2,swin,tiny,3407,56,56
3,swin,tiny,3407,378,378
4,swin,tiny,3407,106,106


## Overall accuracy, precision, recall, and F1 score for the models

In [ ]:
# ["model_name", "model_version", "seed", "accuracy", "precision", "recall", "f1"]
df_metric = []
for model_name in df["model_name"].unique():
    df_model: pd.DataFrame = df[df["model_name"] == model_name]
    for arch in df_model["model_version"].unique():
        df_arch: pd.DataFrame = df_model[df_model["model_version"] == arch]
        for seed in df_arch["seed"].unique():
            df_seed: pd.DataFrame = df_arch[df_arch["seed"] == seed]
            y_true = df_seed["y_true"].tolist()
            y_pred = df_seed["y_pred"].tolist()
            df_metric.append(
                [
                    model_name,
                    arch,
                    seed,
                    metrics.accuracy_score(y_true, y_pred),
                    metrics.precision_score(y_true, y_pred, average="macro", zero_division=0),
                    metrics.recall_score(y_true, y_pred, average="macro", zero_division=0),
                    metrics.f1_score(y_true, y_pred, average="macro", zero_division=0),
                ]
            )

df_metric = pd.DataFrame(
    df_metric,
    columns=[
        "model_name",
        "model_version",
        "seed",
        "accuracy",
        "precision",
        "recall",
        "f1",
    ],
)


In [8]:
df_metric = df_metric.sort_values( # type: ignore
    by=["model_name", "model_version", "seed"],
    ascending=True,
).reset_index(drop=True)
df_metric.head()

,model_name,model_version,seed,accuracy,precision,recall,f1
0,convnext,tiny,0,0.924131,0.923933,0.927091,0.921557
1,convnext,tiny,21,0.924395,0.926500,0.927603,0.923544
2,convnext,tiny,42,0.922908,0.928245,0.927472,0.923721
3,convnext,tiny,84,0.924313,0.925900,0.928410,0.923276
4,convnext,tiny,168,0.925172,0.926248,0.927093,0.922846


### Table

In [9]:
model_info = pd.read_csv("model_info.csv")
model_info_map = {
    r[0]: {
        "n_params": r[1],
        "flops": r[2],
    }
    for r in model_info.values
}
model_info_default = {
    "n_params": f"{0:.2f} M",
    "flops": f"{0:.2f} G",
}

In [10]:
table_headers = [
    "Model",
    "#Params",
    "FLOPs",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "Accuracy std",
    "Precision std",
    "Recall std",
    "F1-score std",
]
rich_table_metric = Table(*table_headers, box=HORIZONTALS)
save_table_metric = []

for model_name in df_metric["model_name"].unique():
    _df0: pd.DataFrame = df_metric[df_metric["model_name"] == model_name]
    for model_version in _df0["model_version"].unique():
        _df1 = _df0[_df0["model_version"] == model_version]
        _df: pd.DataFrame = _df1[["accuracy", "precision", "recall", "f1"]]
        _mean = _df.mean(axis=0).values.ravel()
        _std = _df.std(axis=0).values.ravel()
        _model_info = model_info_map.get(f"{model_name}-{model_version}", model_info_default)
        _n_params = _model_info["n_params"]
        _flops = _model_info["flops"]

        _row = [
            f"{model_name}-{model_version}",
            _n_params,
            _flops,
            _mean[0],
            _mean[1],
            _mean[2],
            _mean[3],
            _std[0],
            _std[1],
            _std[2],
            _std[3],
        ]
        save_table_metric.append(_row)

        rich_table_metric.add_row(
            _row[0],
            f"{_row[1]:.2f} M",
            f"{_row[2]:.2f} G",
            f"{_mean[0] * 100:.2f}%",
            f"{_mean[1] * 100:.2f}%",
            f"{_mean[2] * 100:.2f}%",
            f"{_mean[3] * 100:.2f}%",
            f"{_std[0] * 100:.2f}%",
            f"{_std[1] * 100:.2f}%",
            f"{_std[2] * 100:.2f}%",
            f"{_std[3] * 100:.2f}%",
        )

save_df_metric = pd.DataFrame(save_table_metric, columns=table_headers)

In [11]:
console = Console(record=True)
console.print(rich_table_metric)

───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
                                                                          Accuracy   Precis…   Recall     F1-sco…  
  Model      #Params   FLOPs     Accuracy   Precisi…   Recall   F1-sco…   std        std       std        std      
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  convnex…   28.32 M   8.91 G    92.38%     92.67%     92.74%   92.31%    0.12%      0.20%     0.14%      0.17%    
  mobilen…   32.15 M   4.34 G    91.79%     92.10%     91.75%   91.58%    0.09%      0.20%     0.19%      0.18%    
  mobilen…   9.27 M    1.65 G    91.90%     92.33%     91.90%   91.78%    0.13%      0.22%     0.23%      0.24%    
  mobilen…   3.33 M    0.37 G    90.26%     91.07%     90.12%   90.13%    0.15%      0.13%     0.20%      0.18%    
  mobilen…   37.32 M   5.10 G    91.94%     92.37%     91.95%   91.79%    0.14%      0.25%     0.23%      0.25%    
  mobilen…   10.63 M   1.93 G    91.76%     92.65%     91.75%   91.84%    0.12%      0.24%     0.26%      0.23%    
  mobilev…   1.28 M    0.72 G    89.90%     89.74%     89.68%   89.26%    0.16%      0.25%     0.29%      0.28%    
  mobilev…   4.73 M    2.82 G    92.28%     92.46%     92.40%   92.08%    0.12%      0.27%     0.20%      0.20%    
  mobilev…   10.33 M   6.30 G    92.66%     92.95%     92.83%   92.57%    0.10%      0.20%     0.17%      0.16%    
  resnet-…   11.51 M   3.63 G    89.65%     90.32%     89.58%   89.50%    0.13%      0.22%     0.15%      0.19%    
  resnet-…   21.62 M   7.33 G    90.48%     91.04%     90.49%   90.35%    0.13%      0.23%     0.20%      0.19%    
  resnet-…   24.85 M   8.18 G    90.98%     91.77%     91.20%   91.13%    0.09%      0.24%     0.16%      0.18%    
  swin-ti…   28.02 M   8.98 G    93.05%     93.47%     93.40%   93.12%    0.06%      0.17%     0.14%      0.14%    
  vit-b      86.30 M   33.70 G   92.85%     93.53%     93.12%   93.00%    0.11%      0.21%     0.14%      0.17%    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [12]:
model_name_order = [
    "mobilevit-050",
    "mobilenet-conv_small",
    "mobilevit-100",
    "mobilenet-conv_medium",
    "mobilenet-hybrid_medium",
    "mobilevit-150",
    "resnet-18",
    "resnet-34",
    "resnet-50",
    "swin-tiny",
    "convnext-tiny",
    "mobilenet-conv_large",
    "mobilenet-hybrid_large",
    "vit-b",
]
save_df_metric = save_df_metric.sort_values("Model", key=lambda column: column.map({v: i for i, v in enumerate(model_name_order)})).reset_index(drop=True)

In [17]:
out_path = DATA_DIR / "export" / f"model_metrics_mean_std_pretrained_{PRETRAINED}.csv"
if not out_path.parent.exists():
    out_path.parent.mkdir(parents=True)
save_df_metric.to_csv(out_path, index=False)
save_df_metric

,Model,#Params,FLOPs,Accuracy,Precision,Recall,F1-score,Accuracy std,Precision std,Recall std,F1-score std
0,mobilevit-050,1.28,0.72,0.899008,0.897448,0.896782,0.892556,0.001628,0.002544,0.002893,0.002769
1,mobilenet-conv_small,3.33,0.37,0.902563,0.910732,0.901223,0.901315,0.001482,0.001336,0.001994,0.001771
2,mobilevit-100,4.73,2.82,0.922825,0.924628,0.924044,0.920831,0.001249,0.002656,0.001995,0.002046
3,mobilenet-conv_medium,9.27,1.65,0.918968,0.923313,0.919045,0.917767,0.001257,0.002241,0.002328,0.002359
4,mobilenet-hybrid_medium,10.63,1.93,0.917570,0.926474,0.917471,0.918372,0.001220,0.002443,0.002608,0.002288
5,mobilevit-150,10.33,6.30,0.926568,0.929500,0.928341,0.925745,0.000973,0.002044,0.001714,0.001594
6,resnet-18,11.51,3.63,0.896528,0.903164,0.895831,0.895036,0.001315,0.002170,0.001476,0.001939
7,resnet-34,21.62,7.33,0.904769,0.910392,0.904902,0.903488,0.001266,0.002272,0.002017,0.001865
8,resnet-50,24.85,8.18,0.909836,0.917746,0.911982,0.911345,0.000942,0.002358,0.001604,0.001805
9,swin-tiny,28.02,8.98,0.930467,0.934662,0.934043,0.931160,0.000644,0.001748,0.001354,0.001427
